In [1]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd
from datetime import date

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [2]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [3]:
d = date.today().strftime("%Y%m%d")

In [4]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [5]:
wf_parcels = r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels'
gflu = r'E:\Tasks\REMM-Manage-Base-Year-Data-2\Inputs\gflu_2025.gdb\gflu_2025'
mag_open_space = r'E:\Tasks\REMM-Manage-Base-Year-Data-2\Inputs\MAG_Centers.gdb\MAG_MeetingPolygons_OpenSpaces_CoordUpdate'

In [6]:
wf_pts_export = arcpy.management.FeatureToPoint(
    in_features=wf_parcels,
    out_feature_class=r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_pts",
    point_location="INSIDE"
)

wf_pts_export_df = pd.DataFrame.spatial.from_featureclass(wf_pts_export[0])
wf_pts_export_df = wf_pts_export_df[['parcel_id', 'building_type_id', 'Tax_Exempt', 'NoBuild', 'SHAPE']].copy()
export = os.path.join(gdb, 'wf_pts')
wf_pts_export_df.spatial.to_featureclass(location=export,sanitize_columns=False)  

# arcpy.management.DeleteField(wf_pts_export, ['parcel_id', 'building_type_id', 'Tax_Exempt', 'NoBuild'], method='KEEP_FIELDS')

'e:\\Tasks\\REMM-Manage-Base-Year-Data-2023\\Scripts\\Outputs\\scratch.gdb\\wf_pts'

# policy override

In [7]:
policy_override = os.path.join(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\REMM_Policy_Override_Polygons.gdb\Policy_Override_Polygons_UTM12N")
policy_override_lyr = arcpy.MakeFeatureLayer_management(policy_override, 'pop_lyr')

In [8]:
# create parcel pts
parcels_pts_lyr = arcpy.MakeFeatureLayer_management(export, 'parcels_pts_lyr')

# excluded some special case parcels
arcpy.SelectLayerByAttribute_management(parcels_pts_lyr, 'NEW_SELECTION', "(building_type_id in (6) or Tax_Exempt = 1 or NoBuild = 1)", invert_where_clause='INVERT')

# what parcel ids were these? (parcel_id NOT IN (13670,17117,13898,13907))

<Result 'parcels_pts_lyr'>

In [9]:
# use spatial join to summarize max dua
target_features = parcels_pts_lyr
join_features = policy_override_lyr
output_features = os.path.join(gdb, "parcels_pts_policy_sj")

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

# max_dua
fieldindex = fieldmappings.findFieldMapIndex('max_dua')
fieldmap = fieldmappings.getFieldMap(fieldindex)
fieldmap.mergeRule = 'Max'
fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# max_far
fieldindex = fieldmappings.findFieldMapIndex('max_far')
fieldmap = fieldmappings.getFieldMap(fieldindex)
fieldmap.mergeRule = 'Max'
fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# year
fieldindex = fieldmappings.findFieldMapIndex('year')
fieldmap = fieldmappings.getFieldMap(fieldindex)
fieldmap.mergeRule = 'Max'
fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# run the spatial join
sj = arcpy.analysis.SpatialJoin(target_features, 
                                join_features, 
                                output_features,
                                'JOIN_ONE_TO_ONE', 
                                "KEEP_COMMON", 
                           fieldmappings,
                           search_radius=None,
                           distance_field_name="",
                           match_fields=None, 
                           match_option="INTERSECT")




In [10]:
parcel_pts_policy_sdf = pd.DataFrame.spatial.from_featureclass(os.path.join(gdb, "parcels_pts_policy_sj"))


parcel_pts_policy_sdf = parcel_pts_policy_sdf[['parcel_id', 'max_dua','max_far','year','type1','type2','type3','type4','type5','type6','type7','type8','Notes','MPO', 'CenterName']].copy()
parcel_pts_policy_sdf.columns = ['parcel_id',  'max_dua','max_far','year','type1','type2','type3','type4','type5','type6','type7','type8','locnote','mponote', 'AreaName']

parcel_pts_policy_sdf.loc[(parcel_pts_policy_sdf['max_dua'] > 1) & (parcel_pts_policy_sdf['max_dua'].isna()==False), 'max_dua'] = parcel_pts_policy_sdf['max_dua'] * .75
parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['max_dua'] < .5, 'max_dua'] = np.nan

parcel_pts_policy_sdf.loc[(parcel_pts_policy_sdf['max_far'] > 1) & (parcel_pts_policy_sdf['max_far'].isna()==False), 'max_far'] = parcel_pts_policy_sdf['max_far'] * .75
parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['max_far'] < .5, 'max_far'] = np.nan

# # activate centers in 2026
# parcel_pts_policy_sdf.loc[(parcel_pts_policy_sdf['year'] == 2023), 'year'] = 2026

In [11]:
# # Spot adjustments to big parcels within policy
# parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['parcel_id'] == 35338, 'max_far'] = .25
# parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['parcel_id'] == 57372, 'max_far'] = .25
# parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['parcel_id'] == 48385, 'max_far'] = .2

In [12]:
# BONUS: check for duplicate entrys
duplicate = parcel_pts_policy_sdf[parcel_pts_policy_sdf.duplicated(subset=['parcel_id', 'year'], keep =False)]
n_duplicates = duplicate.shape[0]

print(f'There were {n_duplicates} duplicates rows in the output.')

if n_duplicates > 0:
    duplicate.to_csv(os.path.join(outputs[0],f'duplicates_{d}.csv'))

There were 0 duplicates rows in the output.


In [14]:
parcel_pts_policy_sdf_no_duplicates = parcel_pts_policy_sdf.drop_duplicates(subset=['parcel_id', 'year'], keep='first')
parcel_pts_policy_sdf_no_duplicates.to_csv(os.path.join(outputs[0], f'zoning_parcels_p_{d}.csv'), index=False)

In [ ]:
# pop_df = pd.DataFrame.spatial.from_featureclass(pop)


In [ ]:
# pop_df[(pop_df['DUA'] =='')  & (pop_df['NonResFAR']=='')]
# sort by dua and non res far combined field?